# LLM API Basics

This notebook explores two ways of interacting with the Gemini API:
1. Direct REST requests using Python `requests`
2. Google's official Python SDK

The purpose is to understand which parts of the HTTP communication
are abstracted by the SDK.

# Rest API

In [1]:
from google.colab import userdata
import requests
import json

In [2]:
MODEL = "gemini-3.5-flash-lite"

## Basic Request

In [3]:
api_key = userdata.get("GEMINI_API_KEY")

In [4]:
url = (
    "https://generativelanguage.googleapis.com/v1beta/"
    f"models/{MODEL}:generateContent"
)

headers = {
    "x-goog-api-key": api_key,
    "Content-Type": "application/json"
}

payload_simple = {
    "contents": [
        {
            "parts": [
                {
                    "text": "Explain what an API is in two sentences."
                }
            ]
        }
    ]
}

In [5]:
response_simple = requests.post(
    url,
    headers=headers,
    json=payload_simple
)

print("Status code:", response_simple.status_code)

response_simple.raise_for_status()
response_simple_json = response_simple.json()

Status code: 200


### Response Inspection

In [6]:
print(response_simple.status_code)

200


In [7]:
print("Top-level keys:", response_simple_json.keys())
print(
    "Finish reason:",
    response_simple_json["candidates"][0]["finishReason"]
)
print(
    "Usage metadata:",
    response_simple_json["usageMetadata"]
)

Top-level keys: dict_keys(['candidates', 'usageMetadata', 'modelVersion', 'responseId'])
Finish reason: STOP
Usage metadata: {'promptTokenCount': 9, 'candidatesTokenCount': 55, 'totalTokenCount': 64, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 9}], 'serviceTier': 'standard'}


### Response parsing


In [8]:
generated_text_simple = (
    response_simple_json["candidates"][0]
    ["content"]["parts"][0]["text"]
)

print("Generated text:")
print(generated_text_simple)

Generated text:
An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate and share data with each other. It acts as a messenger that takes a request from a user, delivers it to a system, and then brings the response back.


## System instruction

In [9]:
# Same url and headers
payload_instruction = {
    "system_instruction":{
        "parts":[
            {
                "text": (
                    "You are an expert in technical knowledge. "
                    "Explain concepts clearly and include one practical example")
            }
        ]
    },
    "contents": [
        {
            "parts": [
                {
                    "text": "Explain what an API is in two sentences."
                }
            ]
        }
    ]
}

In [10]:
response_instruction = requests.post(
    url,
    headers=headers,
    json=payload_instruction
)

In [11]:
response_instructions_json = response_instruction.json()

In [12]:
generated_text_instruction = (
    response_instructions_json["candidates"][0]
    ["content"]["parts"][0]["text"]
)

print("Generated text:")
print(generated_text_instruction)

Generated text:
An **API** (Application Programming Interface) is a set of rules and protocols that allows different software applications to communicate and share data with each other. It acts as a messenger that takes your request, tells a system what to do, and then returns the response back to you.

### Practical Example
When you use a travel booking website like Expedia to search for flights, the website uses an **airline’s API** to ask for real-time seat availability and pricing. The airline's system processes the request and sends the data back through the API, displaying the live flight options instantly on your screen without you having to visit every individual airline's website.


## Generation parameters

In [13]:
payload_parameters = {
    "system_instruction":{
        "parts":[
            {
                "text": (
                    "You are an expert in technical knowledge. "
                    "Explain concepts clearly and include one practical example")
            }
        ]
    },
    "contents": [
        {
            "parts": [
                {
                    "text": "Explain what an API is."
                }
            ]
        }
    ],
    "generationConfig": {
        "temperature": 0.2,
        "maxOutputTokens": 150
    }
}

In [14]:
response_parameters = requests.post(
    url,
    headers=headers,
    json=payload_parameters
)

In [15]:
response_parameters_json = response_parameters.json()

generated_text_parameters = (
    response_parameters_json["candidates"][0]
    ["content"]["parts"][0]["text"]
)

print("Generated text:")
print(generated_text_parameters)

Generated text:
### What is an API?

**API** stands for **Application Programming Interface**. 

Think of an API as a **waiter in a restaurant**. 
1. You (the customer/client) are sitting at a table looking at the menu, and you want to order food (data/services). 
2. You cannot go into the kitchen (the server/database) yourself. 
3. The waiter (the API) takes your order, walks back to the kitchen, tells the chef what you want, waits for the food to be prepared, and brings it back to your table.

In technical terms, an API is a set of rules and protocols that allows different software applications to communicate


# Gemini Python SDK

In [16]:
from google import genai
from google.genai import types
api_key = userdata.get("GEMINI_API_KEY")

In [17]:
client = genai.Client(api_key=api_key)

## Basic Request



In [18]:
response_sdk_simple = client.models.generate_content(
    model=MODEL,
    contents="Explain what an API is in two sentences."
)

### Response Inspection

In [19]:
print("Response type")
print(type(response_sdk_simple))

print("\nUsage metadata:")
print(response_sdk_simple.usage_metadata)

Response type
<class 'google.genai.types.GenerateContentResponse'>

Usage metadata:
cache_tokens_details=None cached_content_token_count=None candidates_token_count=57 candidates_tokens_details=None prompt_token_count=10 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=10
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=67 traffic_type=None


### Response Parsing

In [20]:
print("Generated text:")
print(response_sdk_simple.text)

Generated text:
An API, or Application Programming Interface, is a set of rules and protocols that allows different software applications to communicate and share data with one another. Think of it as a waiter in a restaurant who takes your order from the kitchen, delivers it to you, and brings back your food.


## System instruction

In [21]:
response_sdk_instruction = client.models.generate_content(
    model=MODEL,
    contents="Explain what an API is in two sentences.",
    config=types.GenerateContentConfig(
        system_instruction=(
            "You are an expert in technical knowledge. "
            "Explain concepts clearly and include one practical example"
        )
    )
)

In [22]:
print("Generated text:")
print(response_sdk_instruction.text)

Generated text:
An API (Application Programming Interface) is a set of rules and protocols that allows different software applications to communicate and share data with each other. It acts as a messenger that takes your request, tells a system what to do, and then delivers the response back to you.

### Practical Example
When you use a travel booking app to search for flights, the app uses an **API** to communicate with the airline's database. Instead of the app building its own database of flight times and prices, the airline's API fetches that live data and displays the results directly on your screen.


## Generation parameters

In [23]:
response_sdk_parameters = client.models.generate_content(
    model=MODEL,
    contents="Explain what an API is.",
    config=types.GenerateContentConfig(
        system_instruction=(
            "You are an expert in technical knowledge. "
            "Explain concepts clearly and include one practical example"
        ),
        temperature = 0.2,
        max_output_tokens = 150

    )
)

In [24]:
print("Generated text:")
print(response_sdk_parameters.text)

Generated text:
### What is an API?

**API** stands for **Application Programming Interface**. 

Think of an API as a **waiter in a restaurant**. 
1. You (the user/client) are sitting at the table looking at the menu (the application). 
2. The kitchen (the server/database) has everything you want to order, but you can’t just walk back into the kitchen and grab your food. 
3. You need a messenger to take your order to the kitchen and bring your food back to you. That messenger is the **API**.

In technical terms, an API is a set of rules and protocols that allows different software applications to communicate with each other.


# Key Takeaways

- REST calls expose the HTTP endpoint, headers, authentication,
  request payload and raw JSON response.
- The Gemini SDK abstracts most of the HTTP communication and
  provides Python objects for requests and responses.
- Both approaches ultimately interact with the same Gemini API.
- System instructions and generation parameters can be configured
  through either interface.